# Inter-Annotator Agreement Analysis

This notebook analyzes the agreement between different annotators on the misinformation dataset. It uses the functions defined in `Interannotator_Analysis.py` to calculate and display agreement scores.

The following steps are performed:
1.  Import necessary libraries and functions from the analysis script.
2.  Load and preprocess the annotated data into a pandas DataFrame.
3.  Calculate overall agreement using **Fleiss' Kappa** for items annotated by all annotators.
4.  Calculate **Cohen's Kappa** for all pairs of annotators to see pairwise agreement.
5.  Display the results in formatted tables.

In [1]:
# Add the 'src' directory to the system path to import our script
import sys
import os
# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the directory containing the script to the Python path
sys.path.append(notebook_dir)

# Import the functions from your script
from Interannotator_Analysis import get_dataframe, calculate_agreement, calculate_pairwise_agreement

# Import pandas for displaying DataFrames
import pandas as pd

# Configure pandas to display all columns
pd.set_option('display.max_columns', None)

print("Setup complete. Functions from Interannotator_Analysis.py are ready to use.")

Setup complete. Functions from Interannotator_Analysis.py are ready to use.


### 2. Load and Preprocess Data

First, we call the `get_dataframe()` function from the script. This function handles finding all the `*annotations.json` files, loading them, and concatenating them into a single, cleaned pandas DataFrame.

In [2]:
# Load the data
df = get_dataframe()

print("DataFrame loaded successfully.")
print(f"Total annotations loaded: {len(df)}")
print(f"Number of unique annotators: {df['Annotator'].nunique()}")
print(f"Number of unique items: {df['ID'].nunique()}")

# Display the first few rows of the loaded data
df.head()

/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed
['/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed/Gemini_annotations.json', '/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed/shiao-li_annotations.json', '/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed/rachelle_annotations.json', '/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed/Jennifer_annotations.json', '/Users/jennifer/git/MDS-CL/COLX_523_misinformation/src/../data/preprocessed/nicole_annotations.json']
DataFrame loaded successfully.
Total annotations loaded: 2033
Number of unique annotators: 5
Number of unique items: 1001


,Annotator,ID,Text,all_caps,exclamation_marks,hedging,adjectives,unk
0,Annotator 0,0,"No Food, No FEMA: Hurricane Michael’s Survivor...",[],[],[],[],[]
1,Annotator 0,1,Would-be looter in Hurricane Michael-ravaged F...,[],[!],"[Would, would]","[trying, enforcement, trying, enforcement]",[]
2,Annotator 0,2,His argument is correct. Hurricane Rita killed...,[],[],[],[argument],[]
3,Annotator 0,3,im praying for all of my friends down in the C...,[],[],[],[praying],[]
4,Annotator 0,4,To all my Texas streamers: PLEASE be safe if t...,[PLEASE],[!],[if],[],[]


### 3. Calculate Fleiss' Kappa (Overall Agreement)

Now, we'll use the `calculate_agreement()` function to compute Fleiss' Kappa. This metric is suitable for measuring agreement among multiple raters (more than two). The function automatically handles the calculation and prints a formatted table of the results.

In [6]:
# Calculate and display Fleiss' Kappa
fleiss_results_df = calculate_agreement(df)


--- Fleiss' Kappa (Overall Agreement) ---
                  Fleiss' Kappa       Agreement  Items
Feature                                               
all_caps                  0.499        Moderate     40
exclamation_marks         0.807  Almost Perfect     40
hedging                   0.492        Moderate     40
adjectives                0.132          Slight     40
unk                       0.440        Moderate     40


### 4. Calculate Cohen's Kappa (Pairwise Agreement)

Next, we compute Cohen's Kappa for every pair of annotators using the `calculate_pairwise_agreement()` function. This gives us a more detailed view of agreement between specific individuals.

In [4]:
# Calculate and display pairwise Cohen's Kappa
pairwise_results_df = calculate_pairwise_agreement(df)


--- Cohen's Kappa (Pairwise Agreement) ---
                   Annotator 0 & Annotator 1  Annotator 0 & Annotator 2  Annotator 0 & Annotator 3  Annotator 0 & Annotator 4  Annotator 1 & Annotator 2  Annotator 1 & Annotator 3  Annotator 1 & Annotator 4  Annotator 2 & Annotator 3  Annotator 2 & Annotator 4  Annotator 3 & Annotator 4
Feature                                                                                                                                                                                                                                                                                        
all_caps                               0.219                      0.284                      0.324                      0.412                      0.708                      0.792                      0.657                      0.652                      0.692                      0.825
exclamation_marks                      0.816                      0.788                     

### 5. Visualize Agreement Scores

Now that we have the agreement scores, we can create charts to visualize them. A bar chart is a good way to compare the Fleiss' Kappa scores across different features.

#### Fleiss' Kappa Scores by Feature

In [11]:
import altair as alt

# Check if the results DataFrame exists and is not empty
if 'fleiss_results_df' in locals() and not fleiss_results_df.empty:
    # Reset index to make 'Feature' a column for Altair
    fleiss_to_plot = fleiss_results_df.reset_index()
    
    # --- FIX: Rename column to remove apostrophe for Altair ---
    fleiss_to_plot = fleiss_to_plot.rename(columns={"Fleiss' Kappa": "Fleiss Kappa"})
    
    # Create the base bar chart
    bars = alt.Chart(fleiss_to_plot).mark_bar().encode(
        x=alt.X('Feature:N', title='Annotation Feature', sort=None),
        # Use the renamed column, but set a clean title for the axis
        y=alt.Y("Fleiss Kappa:Q", title="Fleiss' Kappa Score"),
        color=alt.Color('Agreement:N', title='Agreement Level'),
        tooltip=[
            alt.Tooltip('Feature'),
            alt.Tooltip("Fleiss Kappa:Q", title="Fleiss' Kappa", format='.3f'),
            alt.Tooltip('Agreement')
        ]
    )
    
    # Create the text layer for labels on top of the bars
    text = bars.mark_text(
        align='center',
        baseline='bottom',
        dy=-5  # Nudge text up so it's not on the bar
    ).encode(
        # Use the renamed column for the text encoding
        text=alt.Text("Fleiss Kappa:Q", format='.3f')
    )
    
    # Layer the bars and text together
    chart = (bars + text).properties(
        title="Fleiss' Kappa Agreement by Feature",
        width=600
    )
    
    display(chart)
else:
    print("Fleiss' Kappa results are not available to plot.")

alt.LayerChart(...)

#### Pairwise Cohen's Kappa Scores

For the pairwise results, a heatmap is an effective way to visualize the agreement matrix between all pairs of annotators for each feature.

We will generate a separate heatmap for each feature.

In [12]:
# Check if the results DataFrame exists and is not empty
if 'pairwise_results_df' in locals() and not pairwise_results_df.empty:
    # --- FIX: Rename column to remove apostrophe for Altair ---
    pairwise_renamed = pairwise_results_df.rename(columns={"Cohen's Kappa": "Cohen Kappa"})

    # Melt the DataFrame to make it suitable for Altair's encoding
    df_melted = pairwise_renamed.reset_index().melt(
        id_vars='Feature', 
        var_name='Annotator Pair', 
        value_name="Cohen Kappa"
    )
    
    # Create the heatmap
    heatmap = alt.Chart(df_melted).mark_rect().encode(
        x=alt.X('Annotator Pair:N', title='Annotator Pairs', sort=None),
        y=alt.Y('Feature:N', title='Feature', sort=None),
        color=alt.Color("Cohen Kappa:Q", 
                        scale=alt.Scale(scheme='redyellowgreen'),
                        legend=alt.Legend(title="Kappa Score")
                       ),
        tooltip=[
            alt.Tooltip('Feature'),
            alt.Tooltip('Annotator Pair'),
            alt.Tooltip("Cohen Kappa:Q", title="Cohen's Kappa", format='.3f')
        ]
    )
    
    # Create the text layer for labels
    text = heatmap.mark_text(baseline='middle').encode(
        text=alt.Text("Cohen Kappa:Q", format='.3f'),
        color=alt.condition(
            # Use white text for dark cells, black for light cells
            alt.datum["Cohen Kappa"] > 0.8, 
            alt.value('white'),
            alt.value('black')
        )
    )
    
    # Layer the heatmap and text
    chart = (heatmap + text).properties(
        title="Heatmap of Pairwise Cohen's Kappa Scores",
        width=700,
        height=300
    )
    
    display(chart)

else:
    print("Pairwise Cohen's Kappa results are not available to plot.")

alt.LayerChart(...)

### 6. Identify and Display Disagreements

To better understand the sources of disagreement, we can create a table that filters for only those items where annotators did not all provide the same rating for a given feature.

The following code will:
1.  Iterate through each unique item (`ID`) in the dataset.
2.  For each item, check every feature to see if there is more than one unique value (i.e., a disagreement).
3.  Collect all items that have at least one disagreement.
4.  Display a table showing the annotations for these specific items, making it easy to see the differences in ratings.

In [9]:
# Find items with any disagreement
disagreement_ids = []
features = ['all_caps', 'exclamation_marks', 'hedging', 'adjectives', 'unk']

# Group by item ID
grouped = df.groupby('ID')

for name, group in grouped:
    has_disagreement = False
    for feature in features:
        # Make a copy to avoid SettingWithCopyWarning
        feature_series = group[feature].copy()
        
        # Check if the column contains lists and convert them to sorted strings if so
        if feature_series.dropna().apply(lambda x: isinstance(x, list)).any():
            # The lambda function handles non-list values (like NaN) gracefully
            feature_series = feature_series.apply(lambda x: str(sorted(x)) if isinstance(x, list) else x)

        # Check if there's more than one unique value for the feature in this group
        if feature_series.nunique() > 1:
            has_disagreement = True
            break # Move to the next item as soon as one disagreement is found
    if has_disagreement:
        disagreement_ids.append(name)

# Filter the original DataFrame to show only the items with disagreements
disagreement_df = df[df['ID'].isin(disagreement_ids)].copy()

# Optional: Sort for clearer presentation
disagreement_df = disagreement_df.sort_values(by=['ID', 'Annotator'])

# To make disagreements more visible, we can pivot the table
# and display it. This puts annotators in columns.
if not disagreement_df.empty:
    print(f"Found {len(disagreement_ids)} items with at least one disagreement.")
    
    # We can show a pivoted view for each feature to highlight disagreements
    for feature in features:
        # Create a temporary copy for this pivot to handle list-to-string conversion
        temp_df = disagreement_df.copy()
        
        # Convert list to string for pivoting if necessary
        if temp_df[feature].dropna().apply(lambda x: isinstance(x, list)).any():
            temp_df[feature] = temp_df[feature].apply(lambda x: str(sorted(x)) if isinstance(x, list) else x)

        pivot_disagreements = temp_df.pivot_table(
            index=['ID', 'Text'], 
            columns='Annotator', 
            values=feature,
            aggfunc='first' # Use first since there's one value per annotator
        )
        # Filter to show only rows (items) that actually have a disagreement for THIS feature
        feature_disagreement_rows = pivot_disagreements.apply(lambda row: row.nunique() > 1, axis=1)
        
        if feature_disagreement_rows.any():
            print(f"\n--- Disagreements for '{feature}' ---")
            display(pivot_disagreements[feature_disagreement_rows])

else:
    print("No disagreements found across any items.")

Found 577 items with at least one disagreement.

--- Disagreements for 'all_caps' ---


,Annotator,Annotator 0,Annotator 1,Annotator 2,Annotator 3,Annotator 4
ID,Text,,,,,
6,"WTF? » No food, no FEMA: Hurricane Michael’s survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw …",[],[],['WTF'],['WTF?'],['WTF']
9,You know they hate you at work when the TV manager picks you to go out with a camera and operate the LIVE DRIVE car in a tornado.,"['DRIVE', 'LIVE']","['DRIVE', 'LIVE']",['LIVE DRIVE'],"['DRIVE', 'LIVE']","['DRIVE', 'LIVE']"
12,President Trump Gives Permission for US Troops to Stay at Trump Hotel in Washington DC https://t.co/PTHILtJqh3 via @gatewaypundit,['DC'],[],[],[],[]
14,"“The effect of Hurricane Matthew on Haiti is catastrophic"" -food + water crisis... http://ln.is/GSphX by #WSJ via @c0nvey",['WSJ'],[],[],[],[]
22,"Drove thru NC last week & experienced my first tornado - went right in front of our vehicle, people on the hwy freaked out more than my kids",['NC'],[],[],[],[]
...,...,...,...,...,...,...
988,I say this every hurricane: it's expensive AF to evacuate & not everyone has a reliable vehicle https://twitter.com/cnalive/status/901460592275275777¬†‚Ä¶,['AF'],[],NaN,NaN,NaN
989,"78-Year-Old Woman, Scared to Drive in Blizzard, Found Dead in Car Outside NJ Burger King http://fb.me/4n2BfuH8G¬†",['NJ'],[],NaN,NaN,NaN
992,The potential for a border war that has been simmering at the Line of Actual Control (LAC) between India and China could reach a boiling point as hostilities increased on Monday night. https://t.co/5oPBrINKZB,['LAC'],[],NaN,NaN,NaN



--- Disagreements for 'exclamation_marks' ---


,Annotator,Annotator 0,Annotator 1,Annotator 2,Annotator 3,Annotator 4
ID,Text,,,,,
1,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!",['!'],['!'],['!'],['!'],[]
4,To all my Texas streamers: PLEASE be safe if the hurricane comes to you! I‚Äôd hate to lose...family...don‚Äôt be a hero. Take shelter. pic.twitter.com/jDPPdMTHwN,['!'],['!'],['!'],['!'],[]
23,*eats the medicine @mlp_twilight gives him* whats a solar flare anyways? does the sun explode?!,['!'],['!'],['!'],[],['!']
48,Lately I been stressing make me wanna put a fuck nigga on a stretcher!,['!'],NaN,[],NaN,['!']
49,Why are so many cars and buses stranded on the highway? Stay off the highway in a blizzard!,['!'],NaN,[],NaN,['!']
55,@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay warm.,"['!', '!']",NaN,[],NaN,[]
58,"Due to the poor air quality from the wildfires, our teams' Ride to Conquer Cancer was cut short yesterday. Big shout out to the Ski Cellar for still hosting our team for a BBQ yesterday so they could celebrate their training and fundraising successes together! #therideABpic.twitter.com/76gLiXP37q",['!'],NaN,[],NaN,['!']
59,Re: Hurricane Matthew: All of the @ASUTennis Student-Athletes are on their way to safety. Thanks to @ASU_Housing for arranging a safe place!,['!'],NaN,[],NaN,['!']
81,Don't Panik! #KelbyTomlinson to the rescue! http://t.co/hujvgsFLUs,"['!', '!']",[],[],NaN,NaN



--- Disagreements for 'hedging' ---


,Annotator,Annotator 0,Annotator 1,Annotator 2,Annotator 3,Annotator 4
ID,Text,,,,,
1,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!","['Would', 'would']",[],"['Would', 'would']",[],['Would']
4,To all my Texas streamers: PLEASE be safe if the hurricane comes to you! I‚Äôd hate to lose...family...don‚Äôt be a hero. Take shelter. pic.twitter.com/jDPPdMTHwN,['if'],[],[],[],[]
15,"i understand everyone has to make a buck, i would encourage you to think if food delivery during a blizzard is the best choice.","['if', 'think', 'would']","['think', 'understand', 'would']","['think', 'understand']","['think', 'understand', 'would']","['think', 'understand', 'would']"
16,"Yeah, but it's also unthinkable to tell hurricane refugees to ""have a great time,"" and to brag about the size of the crowd at shelter.",['about'],[],['unthinkable'],[],[]
17,Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.com/p/BBS2eYWgTbj/¬†,['usual'],[],[],[],[]
...,...,...,...,...,...,...
987,i swea it feels like im about to explode ??,"['about', 'feels', 'like']",[],NaN,NaN,NaN
990,@spinningbot Are you another Stand-user? If you are I will have to detonate you with my Killer Queen.,['If'],[],NaN,NaN,NaN
992,The potential for a border war that has been simmering at the Line of Actual Control (LAC) between India and China could reach a boiling point as hostilities increased on Monday night. https://t.co/5oPBrINKZB,['could'],[],NaN,NaN,NaN



--- Disagreements for 'adjectives' ---


,Annotator,Annotator 0,Annotator 1,Annotator 2,Annotator 3,Annotator 4
ID,Text,,,,,
1,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!","['enforcement', 'enforcement', 'trying', 'tryi...",[],[],[],[]
2,"His argument is correct. Hurricane Rita killed over 100 people on the road, drowned them in their own cars. pic.twitter.com/kulwIZW2EI",['argument'],[],[],[],[]
3,im praying for all of my friends down in the Caribbean who have no where else to go and are forced to ride through the hurricane. be strong https://twitter.com/ttrogdon/status/905205124699750401¬†‚Ä¶,['praying'],[],[],[],[]
5,RT @Stacy_Spencer: There is a Tornado warning. Service is canceled. Get to shelter and be safe. ¬´ praying for memphis.,"['praying', 'warning']",[],[],[],[]
7,"The aftermath of a hurricane is horrific. The heat/humidity is excruciating, no water/ice, no bathing, complete darkness, bugs, no warm food","['bathing', 'darkness', 'excruciating']",['excruciating'],"['darkness', 'excruciating']",['excruciating'],['excruciating']
...,...,...,...,...,...,...
995,"Kashmiris and Muslims protesting in Southall London UK after Namaz Juma agaisnt India's violation of Article 370 and called it ""another Palestine alike in the making."" https://t.co/JepImDzd2E","['making', 'protesting']",[],NaN,NaN,NaN
996,"THR: DESTRUCTIVE TORNADO RIPPING THROUGH TUSCALOOSA, AL NOW! TAKE SHELTER NOW!!! #severe http://dlvr.it/PwRS5 (BN) #tcot",['RIPPING'],[],NaN,NaN,NaN
997,A two year old video is viral as Indians in Spain now are celebrating building of Ram Mandir in Ayodhya.\nA video is viral on social media in which a group of boys and girls in India attire using Dhols and other Indian musical instruments while walking on.. https://t.co/bWhO3OS4De https://t.co/dtd2CosbPe,"['building', 'celebrating', 'using', 'walking']",[],NaN,NaN,NaN



--- Disagreements for 'unk' ---


,Annotator,Annotator 0,Annotator 1,Annotator 2,Annotator 3,Annotator 4
ID,Text,,,,,
6,"WTF? » No food, no FEMA: Hurricane Michael’s survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw …",['WTF'],[],['WTF'],['WTF?'],['WTF']
26,"Cars melt, power down as wildfire turns California town into ‚Äòburning¬†hell‚Äô https://www.siasat.com/news/cars-melt-power-down-wildfire-turns-california-town-burning-hell-1430954/¬†‚Ä¶pic.twitter.com/MEE6HbXWCs","['hell', 'hell']",[],[],[],[]
48,Lately I been stressing make me wanna put a fuck nigga on a stretcher!,['fuck'],NaN,[],NaN,"['fuck', 'nigga']"
68,We're ready for the hurricane and Stephen just left like tf are you doing????,[],NaN,['tf'],NaN,NaN
80,"23 Killed, Cars Melt, Power Down as Wildfire Turns California Town Into ‚ÄòBurning¬†Hell‚Äô https://en.dhwanionline.com/23-killed-cars-melt-power-down-as-wildfire-turns-california-town-into-burning-hell/¬†‚Ä¶","['Hell', 'hell']",NaN,[],NaN,NaN
83,Grocery employees in DC region are working their asses off to make sure folks have food/supplies ahead of blizzard. MVPs.\n\n#blizzard2016,[],['asses'],[],NaN,NaN
89,Yay now I get to ride my bike to school through ass crack of hurricane Matthew. I can hear wind howling from inside my house.,['ass'],[],[],NaN,NaN
156,Dear food lion fuckers: give me a hard time today and you will be under the blizzard,[],['fuckers'],['fuckers'],NaN,NaN
374,‚Äú@itsbritany_btch: Someone give me a ride home from school :(‚Äù omg me too . Not trying to walk home during this tornado watch lmao,['omg'],NaN,NaN,NaN,[]
